In [26]:
# auto reload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
# from wilds import get_dataset
# from wilds.common.data_loaders import get_train_loader

import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt

import numpy as np
from prettytable import PrettyTable
from itertools import chain
from pathlib import Path
import collections
import json
import time
import csv


import torch
import pandas as pd
from tqdm import tqdm
from sconf import Config
from torch import nn
import collections


import os
import logging
import sys
import copy

from PIL import Image
from torch import optim
from transformers import CLIPProcessor, CLIPModel
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

import timm




In [28]:
print(os.path.abspath('../'))
sys.path.append(os.path.abspath('../'))

/home/user01/dg


In [29]:
import Prompt.load_prompts as pt

In [30]:
# from domainbed import algorithms
# from domainbed import networks
# from domainbed.algorithms import Algorithm
from domainbed.optimizers import get_optimizer
# from domainbed import hparams_registry

# from domainbed.lib import wide_resnet
from domainbed.networks.backbones import get_backbone # get backbone used for knowledge distillation.
from domainbed.lib import misc
from domainbed.lib.query import Q     ## query

from domainbed.lib.logger import Logger


sys.path.append(os.path.abspath('../'))
from datasets import MultipleDomainDataset




In [31]:
sys.path.append(os.path.abspath('./'))
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disable tokenizers parallelism


In [32]:
# from PlipTrain import VIT
import PlipTrain

In [33]:
# initialize the args dict
args = {} 


In [34]:
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
    print("Current Device:", torch.cuda.current_device())
else:
    print("CUDA not detected.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args['device'] = device

CUDA Available: True
Device Name: NVIDIA GeForce RTX 4090
CUDA Version: 12.4
Current Device: 0


In [35]:
# dataset = get_dataset(dataset="camelyon17", download=False,root_dir = root_dir)
# print(dataset)
# Transform


# # Get the training, validation, and test sets
# train_data = dataset.get_subset(
#     "train",
#     transform=transform,
# )
# val_data = dataset.get_subset(
#     "val",
#     transform=transform,
# )
# test_data = dataset.get_subset(
#     "test",
#     transform=transform,
# )


# batch_size = 128

# train_loader = get_train_loader("standard", train_data, batch_size=batch_size)
# val_loader = get_train_loader("standard", val_data, batch_size=batch_size)
# test_loader = get_train_loader("standard", test_data, batch_size=batch_size)

In [36]:
root_dir= "/home/user01/data"
args['data_dir'] = root_dir


transform = transforms.Compose(
    [
      transforms.Resize(224),
      transforms.ToTensor(),
    ]
)


root_data_dir = args['data_dir'] + '/camelyon17d'

print(root_dir)
train_dataset = MultipleDomainDataset(root_dir=root_data_dir, domains=['0', '1', '2'], transform=transform)
val_dataset = MultipleDomainDataset(root_dir=root_data_dir, domains=['3'], transform=transform)
test_dataset = MultipleDomainDataset(root_dir=root_data_dir, domains=['4'],  # Specify domains to include
    transform=transform
)

batch_size = 128
# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)


# Optional: Retrieve class and domain mappings
class_mapping = train_dataset.get_class_mapping()
domain_mapping = val_dataset.get_domain_mapping()
print("Class mapping:", class_mapping)
print("Domain mapping:", domain_mapping)


/home/user01/data
Class mapping: {'0': 0, '1': 1}
Domain mapping: {'3': 0}


In [37]:
print(next(iter(train_loader))[0].shape)
sample_image = train_dataset[0][0]


torch.Size([128, 3, 224, 224])


In [38]:
dims = {
    "RN50": 1024,
    "RN101": 512,
    "ViT-B/32": 512,
    "ViT-B/16": 512,
    "ViT-L/14": 768,
}

In [39]:
# Load the full dataset, and download it if necessary
# keys = ["config.yaml"]  #+ .configs  # Include default config file if needed

# keys = [open(key, encoding="utf8") for key in keys]
# args = Config(*keys, default=args)

args['name'] = "PLIP-VL2V-ADiP_prompt0"
args['gpu_id'] = 0
args['dataset_name'] = "camelyon17"
args['lambda'] = 0.5
args['prompt_sytle'] = 2
args['backbone'] = "ViT-B/16"    # "ViT-L/14" "vit-B/16"
args['model'] = "vit-base"
args['pretrained'] = True  # student model must preserve its pretrained weights

args['pretrained_path'] =  Path("./train_output") / args['dataset_name'] / args['name'] / 'models' 
# args["model"] = used for resnet

lr_list = np.array([5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3])

# args["optimizer"] = 'adamw'
args["optimizer"] = ("adam", "adam")
random_state = np.random.RandomState(0)
# args["lr"] = (5e-5, random_state.choice(lr_list))
args["lr"] = 5e-5
# args["weight_decay"] = (0.0, 10 ** random_state.uniform(-6, -2))
args["weight_decay"] = 10 ** random_state.uniform(-6, -2)
args['nsteps'] =  3000

args["work_dir"] = '.'
output_dir = args["work_dir"] / Path("train_output") 
output_dir.mkdir(exist_ok=True, parents=True)

args["out_root"] = args["work_dir"] / Path("train_output") / args["dataset_name"]
args["out_root"].mkdir(exist_ok=True, parents=True)

args["out_dir"] = args["out_root"] / args["name"]
args["out_dir"].mkdir(exist_ok=True, parents=True)

args["model_save"] = True
args["save_step"] = 200

In [40]:
# initialize data dict
data = {}
data['name'] = "camelyon17"
data['num_classes'] = 2
data['num_domains'] = 5
data['class_names'] = ['healthy', 'canserous']
print(sample_image.shape)
data['input_shape'] = sample_image.shape
# data['train_doms'] = [[0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3]]
# data['val_doms'] = [3, 2, 1, 0]
data['train_doms'] = [0, 1, 2]
data['val_doms'] = [3]
data['test_doms'] = [4]


torch.Size([3, 224, 224])


In [ ]:
# # checkpoint = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specific_test/domain0/output_kg/2025-02-07_13-50-00/best_prompt_learner.pth")  # Change the filename if needed

# num_context_tokens = 4
# # agg_vector = text_features_ems


# classnames=["normal", "tumor"]
# plip = CLIPModel.from_pretrained("vinid/plip")
# plip_processor = CLIPProcessor.from_pretrained("vinid/plip")
        

# # # Create the new instance
# # prompt_learner = pt.PromptLearner(classnames, plip, plip_processor, num_context_tokens)

# # # Step 3: Load the saved state_dict
# # prompt_learner.load_state_dict(checkpoint)

# # # Step 4: Move to the correct device if needed
# # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# # prompt_learner.to(device)

# # print("PromptLearner successfully loaded!")


# # prompts = prompt_learner()

# # tokenized_prompts = prompt_learner.tokenized_learnable_prompts

# # # prompt_learner.agg_vector # what is this???

# # text_encoder = pt.TextEncoder(plip)
# # text_features_prompts = text_encoder(prompts, tokenized_prompts)
# # print(text_features_prompts.size())
# # print(text_features_prompts)


# domain_0_dir = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specific_test/domain0/output_kg/2025-02-07_13-50-00/best_prompt_learner.pth")  # Change the filename if needed


# # Create the new instance
# # prompt_learner_0 = pt.PromptLearner(classnames, plip, plip_processor, num_context_tokens, agg_vector=agg_vector, agg_base = True)

# prompt_learner_0 = pt.PromptLearner(classnames, plip, plip_processor, num_context_tokens)

# # Step 3: Load the saved state_dict
# prompt_learner_0.load_state_dict(domain_0_dir)

# # Step 4: Move to the correct device if needed
# prompt_learner_0.to(device)

# print("PromptLearner 0 successfully loaded!")



# domain_1_dir = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specific_test/domain1/output_kg/2025-02-07_13-53-51/last_prompt_learner.pth")  # Change the filename if needed


# # Create the new instance
# prompt_learner_1 = pt.PromptLearner(classnames, plip, plip_processor, num_context_tokens)

# # Step 3: Load the saved state_dict
# prompt_learner_1.load_state_dict(domain_0_dir)

# # Step 4: Move to the correct device if needed
# prompt_learner_1.to(device)

# print("PromptLearner 1 successfully loaded!")


# domain_3_dir = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specific_test/domain3/output_kg/2025-02-07_13-57-39/last_prompt_learner.pth")  # Change the filename if needed


# # Create the new instance
# prompt_learner_3 = pt.PromptLearner(classnames, plip, plip_processor, num_context_tokens)

# # Step 3: Load the saved state_dict
# prompt_learner_3.load_state_dict(domain_0_dir)

# # Step 4: Move to the correct device if needed
# prompt_learner_3.to(device)

# print("PromptLearner 2 successfully loaded!")



# domain_4_dir = checkpoint = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specific_test/domain4/output_kg/2025-02-07_13-59-12/last_prompt_learner.pth")  # Change the filename if needed
#   # Change the filename if needed

# # Create the new instance
# prompt_learner_4 = pt.PromptLearner(classnames, plip, plip_processor, num_context_tokens)

# # Step 3: Load the saved state_dict
# prompt_learner_4.load_state_dict(domain_0_dir)

# # Step 4: Move to the correct device if needed
# prompt_learner_4.to(device)



# prompt_0 = prompt_learner_0()
# token_0 = prompt_learner_0.tokenized_learnable_prompts

# prompt_1 = prompt_learner_1()
# token_1 = prompt_learner_1.tokenized_learnable_prompts

# prompt_3 = prompt_learner_3()
# token_3 = prompt_learner_3.tokenized_learnable_prompts


# prompt_4 = prompt_learner_4()
# token_4 = prompt_learner_4.tokenized_learnable_prompts


# text_encoder = pt.TextEncoder(plip)
# text_features_prompts_0 = text_encoder(prompt_0, token_0)
# text_features_prompts_1 = text_encoder(prompt_1, token_1)
# text_features_prompts_3 = text_encoder(prompt_3, token_3)
# text_features_prompts_4 = text_encoder(prompt_4, token_4)

# learnable_prompts = torch.stack([text_features_prompts_0, text_features_prompts_1, text_features_prompts_3, text_features_prompts_4], dim=0)
# print(learnable_prompts.size())


# mean_learnable_prompts = learnable_prompts.mean(dim=0).detach()
# print(mean_learnable_prompts.size())



# mean_learnable_prompts.requires_grad = False
# mean_learnable_prompts /= mean_learnable_prompts.norm(dim=-1, keepdim=True)

from Prompts. import get_learned_prompts


/home/user01/miniconda3/envs/dg/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/user01/miniconda3/envs/dg/lib/python3.10/site-packages/transformers/modeling_utils.py:399: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly al

Initializing domain-specific contexts
handcraft_base
Initializing PromptLearner without agg_vector.
Tokenized full prompts shape: torch.Size([2, 7, 512])
PromptLearner 0 successfully loaded!


/tmp/ipykernel_272313/2114136311.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  domain_1_dir = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specific_test/domain

Initializing domain-specific contexts
handcraft_base
Initializing PromptLearner without agg_vector.
Tokenized full prompts shape: torch.Size([2, 7, 512])
PromptLearner 1 successfully loaded!


/tmp/ipykernel_272313/2114136311.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  domain_3_dir = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specific_test/domain

Initializing domain-specific contexts
handcraft_base
Initializing PromptLearner without agg_vector.
Tokenized full prompts shape: torch.Size([2, 7, 512])
PromptLearner 2 successfully loaded!


/tmp/ipykernel_272313/2114136311.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  domain_4_dir = checkpoint = torch.load("/home/user01/dg/Prompt/KG/KgCoOp/domain_specifi

Initializing domain-specific contexts
handcraft_base
Initializing PromptLearner without agg_vector.
Tokenized full prompts shape: torch.Size([2, 7, 512])
torch.Size([4, 2, 512])
torch.Size([2, 512])


In [42]:
class ClassificationHead(torch.nn.Linear):
    def __init__(self, normalize, weights, biases=None):
        output_size, input_size = weights.shape
        super().__init__(input_size, output_size)
        self.normalize = normalize
        if weights is not None:
            self.weight = torch.nn.Parameter(weights.clone())
        if biases is not None:
            self.bias = torch.nn.Parameter(biases.clone())
        else:
            self.bias = torch.nn.Parameter(torch.zeros_like(self.bias))

    def forward(self, inputs):
        if self.normalize:
            inputs = inputs / inputs.norm(dim=-1, keepdim=True)
        return super().forward(inputs)

In [43]:
class PLIP:
    """PLIP interface to save memory during algorithm save"""

    def __init__(self, hparams, prompts_embedding = None):
        # Use GPU if available, otherwise CPU
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Device is: ", self.device)
        self.hparams = hparams

        # Load the PLIP model and processor
        self.plip_model = CLIPModel.from_pretrained("vinid/plip")
        self.processor = CLIPProcessor.from_pretrained("vinid/plip")
        
        self.plip_model = self.plip_model.to(self.device)
        self.plip_model.eval()
        for param in self.plip_model.parameters():
            param.requires_grad = False

        self.learned_prompt_embeddings = prompts_embedding

    def get_img_feat(self, x):
        """Get normalized image embeddings"""

        with torch.no_grad():
            clip_img_feat = self.plip_model.get_image_features(x)
            clip_img_feat /= clip_img_feat.norm(dim=-1, keepdim=True)
        return clip_img_feat

    def get_txt_feat(self, labels):
        """Get normalized text embeddings"""

        with torch.no_grad():
            clip_txt_feat = []
            for i in range(labels.size(0)):
                feat = self.zeroshot_weights[labels[i].item()]
                clip_txt_feat.append(feat)
            clip_txt_feat = torch.stack(clip_txt_feat, dim=0)
        return clip_txt_feat

    def get_zeroshot_classifier(self, classnames, templates):

        # logit_scale = self.plip_model.config.logit_scale
        logit_scale = self.plip_model.logit_scale

        print("Getting zeroshot weights.")
        with torch.no_grad():
            # zeroshot_weights = []
            # for classname in tqdm(classnames):
            #     texts = [
            #         template.format(class_name=classname) for template in templates
            #     ]

            #     # Process text inputs for embeddings
            #     inputs = self.processor(
            #         text=texts, images=None, return_tensors="pt", padding=True
            #     ).to(self.device)


            #     embeddings = self.plip_model.get_text_features(**inputs)  # Embed with text encoder
            #     embeddings /= embeddings.norm(dim=-1, keepdim=True)
            #     embeddings = embeddings.mean(dim=0, keepdim=True)
            #     embeddings /= embeddings.norm()
            #     zeroshot_weights.append(embeddings)


            # # Compute zero-shot weights
            # zeroshot_weights = torch.stack(zeroshot_weights, dim=0).to(self.device)
            # print('---------------------------------------- zeroshot0: ' , zeroshot_weights.shape , ' -----------------------')

            # # zeroshot_weights = self.learned_prompt_embeddings
            # zeroshot_weights = torch.transpose(zeroshot_weights, 0, 2)

            # print('---------------------------------------- zeroshot1: ' , zeroshot_weights.shape , ' -----------------------')

            # zeroshot_weights *= logit_scale.exp()
            # print(logit_scale.exp())
            # zeroshot_weights = zeroshot_weights.squeeze().float()
            # print('---------------------------------------- zeroshot2: ' , zeroshot_weights.shape , ' -----------------------')

            # zeroshot_weights = torch.transpose(zeroshot_weights, 0, 1)
            # print('---------------------------------------- zeroshot3: ' , zeroshot_weights.shape , ' -----------------------')
            zeroshot_weights = self.learned_prompt_embeddings
            zeroshot_weights *= logit_scale.exp()

        classification_head = ClassificationHead(
            normalize=True, weights=zeroshot_weights
        )
        self.zeroshot_weights = zeroshot_weights
        return classification_head, zeroshot_weights


In [ ]:
# plip_model = PLIP(args, mean_learnable_prompts)

# plip_model.get_zeroshot_classifier(['normal', 'tumor'], templates = ["a photo of a {class_name}"])
# # # # student = networks.Featurizer()
# # # alg_stage1 = PlipTrain.KnowledgeDistillation(plip_model, data['input_shape'], args, data)  #  def __init__(self, input_shape, args, data): # remove input_shape or not?????


Device is:  cuda
Getting zeroshot weights.


(ClassificationHead(in_features=512, out_features=2, bias=True),
 tensor([[ 4.4592,  1.7208,  0.5827,  ..., -4.8962, -0.2025, -1.8958],
         [ 1.0443, -1.7590,  0.0650,  ..., -0.2881,  1.0725,  3.0099]],
        device='cuda:0'))

In [45]:
# plip_model.get_zeroshot_classifier()

In [46]:
def save_records(records, output_root, filename="records.csv"):
    """
    Save the `records` to a CSV file.

    Args:
        records (list): The training/evaluation records.
        output_dir (str or Path): Directory to save the file.
        filename (str): Name of the output CSV file.
    """
    # Ensure output directory exists
    # output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    # Extract keys from the first record (assuming all records have the same keys)
    keys = records[0].keys()

    # Save to CSV file
    output_path = output_root / filename
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(records)

    print(f"Records saved to {output_path}")

In [47]:
train_doms = [["0", "1", "2"], ["0", "1", "3"], ["0", "2", "3"], ["1", "2", "3"]]
val_doms = ["3", "2", "1", "0"]


transform = transforms.Compose(
    [
    transforms.Resize(224),
    transforms.ToTensor(),
    ]
)

logger = Logger.get(args["out_dir"] / "log.txt")


# plip_model = algorithms.PLIP(args)


batch_size = 128

def train_all_plip(plip_model, args, data, train_doms, val_doms, transform, batch_size, checkpoint_freq, logger=None):
    """
    Shuffle domains and train the model on different train domains.

    Args:
        args (dict): Configuration arguments.
        data (dict): Dataset metadata.
        train_doms (list of lists): List of train domain combinations.
        val_doms (list): List of validation domains.
        transform (torchvision.transforms): Transformations for the dataset.
        checkpoint_freq (int): Frequency for saving checkpoints.
        logger (optional): Logger for logging training progress.
    """

    root_data_dir = args['data_dir'] + '/camelyon17d'

    root_dir= "/home/user01/data"
    args['data_dir'] = root_dir
    args['nsteps'] =  300
    args['save_step'] = 50


    # Iterate through shuffled train domains
    for _index, (train_domains, val_domain) in enumerate(zip(train_doms, val_doms)):

        data['train_doms'] = train_domains
        data['val_doms'] = val_domain

        for i_stage in range(0, 2):
            stage = i_stage + 1

            print(f"Stage {stage}: Training on domains {train_domains}, Validating on domain {val_domain}")

            root_data_dir = args['data_dir'] + '/camelyon17d'

            print(root_dir)
            train_dataset = MultipleDomainDataset(root_dir=root_data_dir, domains=train_domains, transform=transform)
            val_dataset = MultipleDomainDataset(root_dir=root_data_dir, domains=val_domain, transform=transform)
            test_dataset = MultipleDomainDataset(root_dir=root_data_dir, domains=['4'], transform=transform)

            # Create DataLoader
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
            test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

            all_records = []
            results = collections.defaultdict(list) #args['nsteps']
            res, records = PlipTrain.train(stage, train_loader=train_loader, val_loader=val_loader, teacher=plip_model, args=args, data=data, checkpoint_freq= checkpoint_freq, logger=logger)
            
            train_dom_names = ''.join(map(str, data["train_doms"]))
            filename = "t_" + train_dom_names + "v_" + data["val_doms"] + f'_stage{stage}' + "records.csv"
            save_path = Path(args["out_root"]) / args["name"] / 'records'
            save_records(records, save_path, filename=filename)

            # # Optional: Retrieve class and domain mappings
            # class_mapping = train_dataset.get_class_mapping()
            # domain_mapping = val_dataset.get_domain_mapping()
            # print("Class mapping:", class_mapping)
            # print("Domain mapping:", domain_mapping)

            print(f"Stage {stage + 1} of training_dom:{i_stage}completed.\n")
            print("the result was: ", res)



In [48]:
plip_model = PLIP(args, mean_learnable_prompts)


Device is:  cuda


In [49]:
train_all_plip(plip_model, args, data, train_doms, val_doms, transform, batch_size, checkpoint_freq = 20, logger=logger)

Stage 1: Training on domains ['0', '1', '2'], Validating on domain 3
/home/user01/data
Prompt Template: a photo of a \{class\}


NAME: vit-base 


cls names:  ['healthy', 'canserous']
Getting zeroshot weights.
==========================self.embed_dim: 512
==========================self.student.n_outputs: 768
INFO 02/10 01:31:01 | # of params = 86193410
INFO 02/10 01:31:01 | ===== stage 1 =====


EVAL: 100%|██████████| 285/285 [00:42<00:00,  6.74it/s]/s]


INFO 02/10 01:31:57 | val_acc     val_precis  val_recall  val_f1      step        epoch       loss        step_time   eval_time  
INFO 02/10 01:31:57 | 0.728993    0.725576    0.523340    0.471248    20          0.008058    -0.139087   0.611570    42.815489  


EVAL: 100%|██████████| 285/285 [00:40<00:00,  7.07it/s]/s]  

INFO 02/10 01:32:47 | 0.822248    0.824116    0.712732    0.740034    40          0.016116    -0.245032   0.526933    40.931958  


...

INFO 02/10 01:58:49 | 0.872229    0.873617    0.877863    0.871992    40          0.018116    -0.264464   0.520626    80.421253  



EVAL: 100%|██████████| 559/559 [01:21<00:00,  6.85it/s]/s]  

INFO 02/10 02:00:20 | 0.871432    0.876935    0.879605    0.871382    60          0.027174    -0.335124   0.497543    82.067487  



Training:  21%|██        | 62/300 [04:52<18:40,  4.71s/it]  


KeyboardInterrupt: 

In [ ]:
train_doms = [["0", "1", "2"]]
val_doms = ["3"]
test_dom = ["4"]
def inference_all_plip(plip_model, args, data, train_doms, val_doms, test_dom, transform, batch_size, logger=None):
    """
    Shuffle domains and train the model on different train domains.

    Args:
        args (dict): Configuration arguments.
        data (dict): Dataset metadata.
        train_doms (list of lists): List of train domain combinations.
        val_doms (list): List of validation domains.
        transform (torchvision.transforms): Transformations for the dataset.
        checkpoint_freq (int): Frequency for saving checkpoints.
        logger (optional): Logger for logging training progress.
    """

    root_data_dir = args['data_dir'] + '/camelyon17d'

    root_dir= "/home/user01/data"
    args['data_dir'] = root_dir

    results = []
    # Iterate through shuffled train domains
    
    for _index, (train_domains, val_domain) in enumerate(zip(train_doms, val_doms)):

        data['train_doms'] = train_domains
        data['val_doms'] = val_domain

        stage = 3

        print(f"Stage {stage}: Training on domains {train_domains}, Validating on domain {val_domain}")

        root_data_dir = args['data_dir'] + '/camelyon17d'

        print(root_dir)
        test_dataset = MultipleDomainDataset(root_dir=root_data_dir, domains=test_dom, transform=transform)

        # Create DataLoader
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

        res, records = PlipTrain.inference(stage, test_loader=test_loader, teacher=plip_model, args=args, data=data, logger=logger)
        results.append(res)

        train_dom_names = ''.join(map(str, data["train_doms"]))
        filename = "t_" + train_dom_names + "v_" + data["val_doms"] + f'_stage{stage}' + "test_records.csv"
        save_path = Path(args["out_root"]) / args["name"] / 'records'
        save_records(records, save_path, filename=filename)

        print(f"Inference of index:{_index} completed.\n")
        print("the result was: ", res)

    # mean_results = defaultdict(list)
    # for res in results:
    #     for key, value in res.items():
    #         mean_results[key].append(value)
    
    # mean_results = {key: sum(values) / len(values) for key, values in mean_results.items()}


    # print("Mean of results:", mean_results)
    # print('aggregated resluts:')
    # for key in mean_results:
    #     logger.info(key + " is: " + str(mean_results[key]))


    # csv_file = "mean_results.csv"
    # with open(csv_file, mode='w', newline='') as file:
    #     writer = csv.DictWriter(file, fieldnames=mean_results.keys())
        
    #     # Write the header
    #     writer.writeheader()
        
    #     # Write the mean values
    #     writer.writerow(mean_results)

    # print(f"Mean results saved to {csv_file}")

    # filename = "aggregated_test_records.csv"
    # save_path = Path(args["out_root"]) / args["name"] / 'records'
    # save_records(mean_results, save_path, filename=filename)


    return res

inference_all_plip(plip_model, args, data, train_doms, val_doms, test_dom, transform, batch_size, logger=logger)

Stage 3: Training on domains ['0', '1', '2'], Validating on domain 3
/home/user01/data
Prompt Template: a photo of a \{class\}


NAME: vit-base 


cls names:  ['healthy', 'canserous']
Getting zeroshot weights.
==========================self.embed_dim: 512
==========================self.student.n_outputs: 768
INFO 02/10 02:05:16 | # of params = 86193410
INFO 02/10 02:05:16 | ===== stage 3 =====
loading model for stage3 and path: train_output/camelyon17/PLIP-VL2V-ADiP_prompt0/models/t_012v_3_stage2_best.pth


/home/user01/dg/VL2V-ADiP/PlipTrain.py:483: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_dict = torch.load(model_path)
/home/user01/dg/VL2V-ADiP/PlipTrain.py:488: Fut

INFO 02/10 02:11:32 | test_acc    test_preci  test_recal  test_f1     eval_time  
INFO 02/10 02:11:32 | 0.946127    0.956244    0.935223    0.943133    375.546369 
INFO 02/10 02:11:32 | ---
INFO 02/10 02:11:32 | acc = 94.613%
INFO 02/10 02:11:32 | test_prec = 95.624%
INFO 02/10 02:11:32 | test_rec = 93.522%
INFO 02/10 02:11:32 | test_f1 = 94.313%
Records saved to train_output/camelyon17/PLIP-VL2V-ADiP_prompt0/records/t_012v_3_stage3test_records.csv
Inference of index:0 completed.

the result was:  {'acc': np.float64(0.9461273513993531), 'test_prec': 0.9562444591462145, 'test_rec': 0.9352226638016505, 'test_f1': 0.9431328233857614}


{'acc': np.float64(0.9461273513993531),
 'test_prec': 0.9562444591462145,
 'test_rec': 0.9352226638016505,
 'test_f1': 0.9431328233857614}

: 

In [ ]:
from domainbed.lib.logger import Logger
logger = Logger.get(args["out_dir"] / "log.txt")

all_records = []
results = collections.defaultdict(list) #args['nsteps']
stage = 1
res, records = train(stage, train_loader=train_loader, val_loader=val_loader, teacher=plip_model, args=args, data=data, checkpoint_freq=5 , logger=logger)


all_records.append(records)
for k, v in res.items():
    results[k].append(v)

In [ ]:
all_records_stage2 = []
results = collections.defaultdict(list) #args['nsteps']
stage = 2

plip_model = algorithms.PLIP(args)

res, records = train(stage, train_loader=train_loader, val_loader=val_loader, teacher=plip_model, args=args, data=data, checkpoint_freq=5 , logger= logger)

all_records_stage2.append(records)
for k, v in res.items():
    results[k].append(v)

In [ ]:
stage = 3
algorithm = VL2V_ADiP(stage, teacher=plip_model, input_shape=data["input_shape"], args=args, data=data)
algorithm.cuda()
# def __init__(self, stage, teacher, input_shape, args, data):

n_params = sum([p.numel() for p in algorithm.parameters()])

# train_minibatches_iterator = zip(*train_loaders)

algorithm = load_algorithm(stage, algorithm, args, logger)


evaluator = Evaluator(
    train_loader = train_loader,
    val_loader = test_loader,
    args = args,
    data = data,
    logger = logger,
    test = True,
)

all_records = []

steps_per_epoch = len(train_loader)


results = {}

eval_start_time = time.time()
accuracies, summaries = evaluator.evaluate(algorithm)
results["eval_time"] = time.time() - eval_start_time

# results = (epochs, loss, step, step_time)
results_keys = (
    # list(summaries.keys())
    list(accuracies.keys())
    + list(results.keys())
)
# merge results
# results.update(summaries)
results.update(accuracies)

logger.info(misc.to_row([key for key in results_keys]))
logger.info(misc.to_row([results[key] for key in results_keys]))


# all_records.append((data['test_doms'], results))

# print('here in else now we should set columns')
# columns = ["Domain"] + ["mean_acc"] + data["class_names"] #changed: added mean acc
# acc_table = PrettyTable(columns)
# for i in range(len(all_records)):
#     name, acc_dict = all_records[i]
#     all_classes = [f"{acc * 100:.2f}" for acc in list(acc_dict.values())]
#     values = [acc for acc in list(acc_dict.values())]
#     mean = [f"{sum(values) / len(values) * 100:.2f}"]
#     row = [name] + mean + all_classes
#     acc_table.add_row(row)

# print(acc_table)

# with open(args["
# 
# "] / "result.csv", "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(acc_table.field_names)
#     # Write the rows
#     for row in acc_table.rows:
#         writer.writerow(row)

In [8]:
# from pathlib import Path
# import csv
# dict1 = {"ali": 1, "reza":2, "maraz":3, "dalghak": 4}
# train_doms = ["1", "2", "3"]
    
# train_dom_names = ''.join(map(str, train_doms))
# filename = "t_" + train_dom_names + "v_" + "4" + "aggregated_test_records.csv"
# save_path = Path('.') / filename

# with open(save_path, mode='w', newline='') as file:
#     writer = csv.DictWriter(file, fieldnames=dict1.keys())
    
#     # Write the header
#     writer.writeheader()
    
#     # Write the mean values
#     writer.writerow(dict1)

# print(f"Mean results saved to {save_path}")